# ML-07 — Baseline Action Score and Top-20 Review

This notebook freezes a transparent review-priority baseline before model training. It uses April 2026 measurements to rank May decline risk; May→June remains sealed. The score orders human review and does not automatically edit, prune, or refresh content.

## 1. My fixed rule and reason codes

The 0–100 score is fixed as 40% log exposure, 30% negative prior momentum, 20% established-content age, and 10% position opportunity around the visible 1–20 range. Missing prior history contributes no momentum risk; invalid position contributes no position opportunity and is flagged for monitoring. Reason codes describe meaningful exposure, prior momentum, visibility, and missing evidence.

In [1]:
from pathlib import Path
import json
import sys

import duckdb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

ROOT = Path.cwd().resolve()
while not (ROOT / 'work' / 'scripts').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from work.scripts.refresh_capstone import BASELINE_WEIGHTS, assign_action, baseline_score, precision_at_k

assert abs(sum(BASELINE_WEIGHTS.values()) - 1.0) < 1e-12
weights = pd.DataFrame({'component': list(BASELINE_WEIGHTS), 'weight': list(BASELINE_WEIGHTS.values())})
display(weights)
print('Reason codes: meaningful_exposure, limited_exposure, negative_prior_momentum, positive_prior_momentum, visible_position, position_unavailable.')

,component,weight
0,exposure,0.4
1,momentum_risk,0.3
2,established_content,0.2
3,position_opportunity,0.1


Reason codes: meaningful_exposure, limited_exposure, negative_prior_momentum, positive_prior_momentum, visible_position, position_unavailable.


## 2. Build the validation ranked queue

All rows are the identical eligible April anchor population. Stable descending sorting preserves input order when scores tie. The complete pseudonymous queue is kept in an ignored CSV; only aggregate metrics are committed.

In [2]:
cache = ROOT / 'work' / 'outputs' / 'model_examples.parquet'
con = duckdb.connect()
validation = con.execute("SELECT * FROM read_parquet(?) WHERE month = DATE '2026-04-01'", [str(cache)]).df()
validation['baseline_score'] = baseline_score(validation)
actions = assign_action(validation, validation['baseline_score'])
queue = pd.concat([validation.reset_index(drop=True), actions.reset_index(drop=True)], axis=1)
queue = queue.sort_values('baseline_score', ascending=False, kind='stable').reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)
queue.to_csv(ROOT / 'work' / 'outputs' / 'baseline_queue.csv', index=False)
p50 = precision_at_k(validation['future_decline'], validation['baseline_score'], 50)
base_rate = float(validation['future_decline'].mean())
recall50 = float(queue.head(50)['future_decline'].sum() / validation['future_decline'].sum())
metrics = {
    'method': 'fixed_transparent_baseline', 'validation_anchor': '2026-04 predicting 2026-05',
    'validation_rows': int(len(validation)), 'validation_positive_rate': base_rate,
    'precision_at_50': float(p50), 'recall_at_50': recall50,
    'lift_at_50': float(p50 / base_rate),
    'average_precision': float(average_precision_score(validation['future_decline'], validation['baseline_score'])),
    'weights': BASELINE_WEIGHTS,
    'thresholds': {'eligibility_impressions': 100, 'decline_ratio': 0.80, 'high_priority_score': 70, 'medium_priority_score': 50},
    'sealed_month_used': False,
}
(ROOT / 'work' / 'outputs' / 'baseline_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
print(f"Validation rows: {len(validation):,}; base rate: {base_rate:.1%}")
print(f"Baseline Precision@50: {p50:.1%}; lift@50: {metrics['lift_at_50']:.2f}x; average precision: {metrics['average_precision']:.3f}")
display(queue['suggested_action'].value_counts().rename_axis('action').to_frame('rows'))
display(queue['confidence_tier'].value_counts().rename_axis('confidence').to_frame('rows'))

Validation rows: 93,474; base rate: 53.8%
Baseline Precision@50: 66.0%; lift@50: 1.23x; average precision: 0.533


,rows
action,
monitor,50434
protect,28983
consolidate_or_prune_review,13351
refresh_or_expand_review,706


,rows
confidence,
low,60905
medium,32557
high,12


## 3. Top-20 review

The review below uses pseudonymous content references only. A pick is wrong for this measured label when the next month did not decline by at least 20%; even a true positive still requires checking intent, seasonality, and business importance before a content action.

In [3]:
top20 = queue.head(20).copy()
top20['observed_validation_result'] = np.where(top20['future_decline'].eq(1), 'measured decline', 'did not meet decline label')
top20['what_could_make_review_wrong'] = np.where(
    top20['future_decline'].eq(1),
    'seasonality, tracking change, or intentional demand shift',
    'fixed score over-weighted exposure, age, or prior movement',
)
review_columns = ['rank', 'content_hash_id', 'baseline_score', 'suggested_action', 'reason_codes', 'confidence_tier', 'observed_validation_result', 'what_could_make_review_wrong']
display(top20[review_columns])
print(f"Top-20 observed precision: {top20.future_decline.mean():.1%}")

,rank,content_hash_id,baseline_score,suggested_action,reason_codes,confidence_tier,observed_validation_result,what_could_make_review_wrong
0,1,content_cd3d932d4e1c8db0,87.504979,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."
1,2,content_5a77dbf5671c5a65,82.877511,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
2,3,content_45be1a7ea8833f6a,81.870161,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."
3,4,content_347886326306b849,81.226159,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."
4,5,content_ef1caed111f80858,81.172838,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
5,6,content_f6116743b00afc2d,80.934003,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."
6,7,content_c2d5b16a59e75cfa,80.725831,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
7,8,content_945d6ff91386c817,80.670249,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."
8,9,content_b3be04da93e2a210,80.541657,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."
9,10,content_34a70fea29d15f24,80.462739,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,high,measured decline,"seasonality, tracking change, or intentional d..."


Top-20 observed precision: 60.0%


## 4. Weak picks and leakage check

Weak picks are false positives or rows with incomplete prior/position context. They show why the baseline is a comparison floor rather than a finished decision system. Outcome fields are used only after ranking for evaluation, and changing them cannot change the score.

In [4]:
weak = top20.loc[top20['future_decline'].eq(0) | top20['prior_impressions'].isna() | ~top20['position_available']].copy()
mutated = validation.assign(future_decline=1-validation['future_decline'], outcome_impressions=0)
assert baseline_score(validation).equals(baseline_score(mutated))
assert validation['month'].eq(pd.Timestamp('2026-04-01')).all()
assert set(BASELINE_WEIGHTS) == {'exposure', 'momentum_risk', 'established_content', 'position_opportunity'}
assert not any(name.startswith(('future_', 'outcome_')) for name in BASELINE_WEIGHTS)
print(f'Weak or incomplete-evidence picks in top 20: {len(weak)}')
print('Leakage check passed: target/outcome changes do not alter the frozen baseline score; sealed rows were not read.')
display(weak[review_columns] if len(weak) else pd.DataFrame({'review': ['No weak pick met the declared checks']}))

Weak or incomplete-evidence picks in top 20: 8
Leakage check passed: target/outcome changes do not alter the frozen baseline score; sealed rows were not read.


,rank,content_hash_id,baseline_score,suggested_action,reason_codes,confidence_tier,observed_validation_result,what_could_make_review_wrong
1,2,content_5a77dbf5671c5a65,82.877511,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
4,5,content_ef1caed111f80858,81.172838,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
6,7,content_c2d5b16a59e75cfa,80.725831,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
11,12,content_757b1fa67827358d,80.233975,refresh_or_expand_review,negative_prior_momentum;visible_position,high,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
13,14,content_c684e826fd2cc874,79.733597,refresh_or_expand_review,negative_prior_momentum;visible_position,medium,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
16,17,content_cb050d62c413425b,79.419872,refresh_or_expand_review,negative_prior_momentum;visible_position,medium,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
17,18,content_9ea1eff7e514b02c,79.091243,refresh_or_expand_review,meaningful_exposure;negative_prior_momentum;vi...,medium,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."
19,20,content_838448bb9f166179,78.968670,refresh_or_expand_review,negative_prior_momentum;visible_position,medium,did not meet decline label,"fixed score over-weighted exposure, age, or pr..."


## Self-check

- [x] The fixed rule, weights, thresholds, and reason codes are explicit.
- [x] The April validation queue is ranked deterministically and reviewed.
- [x] Precision@50 is measured on the eligible validation population.
- [x] Future fields cannot alter the score and May→June remains sealed.
- [x] Recommendations are human-review-only and use careful directional language.